In [1]:
import pandas as pd

In [6]:
df_rera = pd.read_parquet("../data/clean_rer_a_2026-11-25.parquet")
df_rerb = pd.read_parquet("../data/clean_rer_b_2026-11-25.parquet")
df_rerd = pd.read_parquet("../data/clean_rer_d_2026-11-25.parquet")
df_incidents = pd.read_parquet("../data/clean_incidents_2026-11-25.parquet")

In [7]:
df_rera.columns

Index(['journey_id', 'line', 'mission', 'origin', 'destination', 'direction',
       'is_long', 'nb_stops', 'total_delay_sec', 'max_delay_added_sec',
       'worst_stop', 'disappeared', 'first_seen', 'last_seen', 'date',
       'stop_name', 'aimed_time', 'expected_time', 'delay_sec', 'platform',
       'status', 'stop_order', 'delay_added_sec', 'retard_min', 'est_supprime',
       'zone_tunnel', 'heure_pointe', 'tranche_horaire'],
      dtype='str')

In [ ]:
# AXE 1: comparaison macro retards B/D vs A

df_all = pd.concat([df_rera, df_rerb, df_rerd], ignore_index=True)
df_valide = df_all[df_all["est_supprime"] == False] # IDFM mets 0 de retard aux trains supprimés

macro = df_valide.groupby("line")["retard_min"].describe(percentiles=[0.5, 0.9, 0.95])
macro = macro[["count", "mean", "50%", "90%", "95%"]].rename(columns={"50%": "median", "90%": "p90", "95%": "p95"})

# on regarde la meme chose mais uniquement sur chatelet qui est commune au 3 (pas de biais de longueur)

df_chatelet = df_valide[df_valide["stop_name"] == "Châtelet-Les Halles"]
macro_chatelet = df_chatelet.groupby("line")["retard_min"].describe(percentiles=[0.5, 0.9, 0.95])
macro_chatelet = macro_chatelet[["count", "mean", "50%", "90%", "95%"]].rename(
    columns={"50%": "median", "90%": "p90", "95%": "p95"}
)

# on regarde sur zone_tunnel (gare du nord et chatelet pour b,d et chatelet pour a)

df_valide.groupby(["line", "zone_tunnel"])["retard_min"].agg(
    mean="mean",
    median="median",
    p90=lambda s: s.quantile(0.90),
    p95=lambda s: s.quantile(0.95),
)

# on regarde uniquement le retard prit sur zone_tunnel

df_valide.groupby(["line", "zone_tunnel"])["delay_added_sec"].agg(["mean", "median", lambda s: s.quantile(0.90)])

# est ce que le retard s'ajoute en gare de sortie en fonction du sens de trajet (gare du nord vers le nord et chatelet vers le sud)

df_tunnel = df_valide[
    df_valide["zone_tunnel"]
    & (df_valide["stop_order"] > 1)
    & df_valide["direction"].isin(["Nord", "Sud", "Est", "Ouest"])
]

df_tunnel.groupby(["line", "direction", "stop_name"])["delay_added_sec"].agg(["mean", "median"])

# aller/retour : trains dont le terminus ne correspondait pas au branches habituelles (ex CDG2 pour le b)
#

mean  median
line  direction stop_name                             
rer_a Est       Châtelet-Les Halles -13.808970    -5.0
      Ouest     Châtelet-Les Halles  -5.467365   -10.0
rer_b Nord      Châtelet-Les Halles  14.235032    11.0
                Gare du Nord         43.304621    22.0
      Sud       Châtelet-Les Halles  45.340346    33.0
                Gare du Nord         -1.442104     0.0
rer_d Nord      Châtelet-Les Halles  34.568307     0.0
                Gare du Nord         48.588075    32.0
      Sud       Châtelet-Les Halles  82.319210    66.0
                Gare du Nord         -3.806086   -10.0

In [19]:
# AXE 1: Suite: comparaison en fonction des heures de la journée

df_valide.groupby(["line", "tranche_horaire"])["retard_min"].mean().unstack()

# Ratio exact Pointe (semaine) vs Heures creuses (semaine)
pointe = df_valide[df_valide["heure_pointe"]].groupby("line")["retard_min"].agg(
    pointe_mean="mean",
    pointe_p90=lambda s: s.quantile(0.90),
)
creuses = df_valide[df_valide["tranche_horaire"] == "Heures creuses"].groupby("line")["retard_min"].agg(
    creuses_mean="mean",
    creuses_p90=lambda s: s.quantile(0.90),
)

comp = pointe.join(creuses)
comp["ecart_mean"] = comp["pointe_mean"] - comp["creuses_mean"]
comp["ratio_mean"] = comp["pointe_mean"] / comp["creuses_mean"]
comp["ratio_p90"] = comp["pointe_p90"] / comp["creuses_p90"]
comp

tunnel = df_valide[
    df_valide["zone_tunnel"]
    & (df_valide["stop_order"] > 1)
    & df_valide["direction"].isin(["Nord", "Sud", "Est", "Ouest"])
]
tunnel.groupby(["line", "tranche_horaire"])["delay_added_sec"].mean().unstack()

tranche_horaire,Heures creuses,Pointe matin,Pointe soir,Soirée,Week-end
line,,,,,
rer_a,-10.647727,-0.704552,-10.113305,-10.445023,-12.679140
rer_b,20.871316,38.214481,57.577642,13.716403,12.211963
rer_d,37.183753,50.451653,68.173888,22.164286,29.066270
